# Hyperparameter Tuning with Optuna + MLflow

This notebook performs **full fine-tuning** of EfficientNet-B3 on the 525 Bird Species dataset. Unlike the baseline (which only trained the classification head), here we train the entire network in two stages:

1. **Warm-up stage** — backbone frozen, only the new classifier head trains. This gives the head sensible weights before exposing the pretrained backbone to large gradients.
2. **Fine-tuning stage** — entire network unfrozen, with separate learning rates for the head and backbone. This lets the model adapt its feature extraction to bird-specific features.

## Hyperparameter Tuning

We use **Optuna** to automatically search for the best combination of hyperparameters across **30 trials**. Each trial runs a complete two-stage training run.

**Tuned hyperparameters:**
- `n_warmup_epochs` — int, 1 to 5
- `lr_head` — float, log scale, 1e-4 to 1e-2
- `lr_backbone` — float, log scale, 1e-5 to 1e-3

**Fixed hyperparameters:**
- Total epochs: **15** (`n_finetune_epochs = 15 - n_warmup_epochs`)
- Optimizer: **RMSprop** (momentum=0.9, alpha=0.9, eps=1e-8)
- Scheduler: **StepLR** during fine-tuning (step_size = `n_finetune_epochs // 3`, gamma=0.1)
- Batch size: 512
- Weight decay: 1e-5

## Experiment Tracking

Each trial is logged as a separate **MLflow run**, capturing:
- All hyperparameters
- Per-epoch metrics (train/val loss, top-1, top-5)
- The training/validation loss curve as a PNG artifact

## Best Models

After all 30 trials, we save two checkpoints:
- **`EN_best_top1.pth`** — model with the highest top-1 validation accuracy
- **`EN_best_top5.pth`** — model with the highest top-5 validation accuracy

The corresponding MLflow runs are tagged with `best_top1: true` and `best_top5: true` so they're easy to identify in the MLflow UI.

## 0. Sync Data from S3

Downloads the dataset from S3 to the SageMaker instance's local storage. This only runs once per session — `aws s3 sync` skips files that already exist.

In [ ]:
import subprocess
from pathlib import Path

S3_BUCKET    = "bird-ml-halajeel"
S3_DATA_PATH = "data/raw/birds-525"
LOCAL_DATA_DIR = Path("./data/raw/birds-525")

LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)

subprocess.run([
    "aws", "s3", "sync",
    f"s3://{S3_BUCKET}/{S3_DATA_PATH}/",
    str(LOCAL_DATA_DIR),
], check=True)

print(f"Data ready at: {LOCAL_DATA_DIR.resolve()}")

## 1. Load Dataset & DataLoaders

In [ ]:
from datasets import load_dataset
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
import pyarrow.parquet as pq

RAW_DATA_DIR = Path("./data/raw/birds-525")

dataset = load_dataset("parquet", data_dir=str(RAW_DATA_DIR / "data"))

In [ ]:
# Labels in the dataset are sparse — build a mapping from original label to dense index
# by scanning all parquet files directly with pyarrow (much faster than going through HuggingFace).
all_labels = set()
for parquet_file in (RAW_DATA_DIR / "data").glob("*.parquet"):
    table = pq.read_table(parquet_file, columns=["label"])
    all_labels.update(table["label"].to_pylist())

label_to_idx = {original: dense for dense, original in enumerate(sorted(all_labels))}
print(f"Built label remapping for {len(label_to_idx)} classes (dense range 0–{len(label_to_idx) - 1})")

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

eval_transforms = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

class BirdDataset(Dataset):
    def __init__(self, hf_split, transform, label_to_idx):
        self.data = hf_split
        self.transform = transform
        self.label_to_idx = label_to_idx

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sample = self.data[idx]
        return self.transform(sample["image"]), self.label_to_idx[sample["label"]]

BATCH_SIZE = 512

train_loader = DataLoader(BirdDataset(dataset["train"],      train_transforms, label_to_idx), batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(BirdDataset(dataset["validation"], eval_transforms,  label_to_idx), batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(BirdDataset(dataset["test"],       eval_transforms,  label_to_idx), batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print("DataLoaders ready.")

## 2. Optuna Hyperparameter Search

Runs 30 trials. Each trial:
1. Samples `n_warmup_epochs`, `lr_head`, `lr_backbone`
2. Builds a fresh EfficientNet-B3 from pretrained weights
3. Stage 1: warmup with backbone frozen
4. Stage 2: full fine-tuning with two parameter groups + StepLR
5. Logs everything to MLflow (params, metrics, loss curve PNG)
6. If this trial beats the running best top-1 or top-5, saves its model checkpoint

After all trials, the best MLflow runs are tagged with `best_top1: true` and `best_top5: true`.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models
import mlflow
import optuna
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

# === Setup ===
NUM_CLASSES  = len(label_to_idx)
WEIGHTS_PATH = Path("./efficientnet_b3_rwightman-b3899882.pth")
MODELS_DIR   = Path("./models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

TOTAL_EPOCHS = 15
N_TRIALS     = 30
WEIGHT_DECAY = 1e-5
RMS_MOMENTUM = 0.9
RMS_ALPHA    = 0.9
RMS_EPS      = 1e-8

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device} | Classes: {NUM_CLASSES}")

criterion = nn.CrossEntropyLoss()


# === Helper functions ===
def top_k_accuracy(outputs, labels, k):
    _, top_k_preds = outputs.topk(k, dim=1)
    correct = top_k_preds.eq(labels.view(-1, 1).expand_as(top_k_preds))
    return correct.any(dim=1).float().mean().item()


def build_fresh_model():
    """Create a new EfficientNet-B3 with pretrained weights and a fresh head."""
    m = models.efficientnet_b3(weights=None)
    state_dict = torch.load(WEIGHTS_PATH, map_location="cpu", weights_only=True)
    m.load_state_dict(state_dict)
    in_features = m.classifier[1].in_features
    m.classifier[1] = nn.Linear(in_features, NUM_CLASSES)
    return m.to(device)


def run_epoch(model, optimizer, loader, training, epoch_label):
    model.train() if training else model.eval()
    total_loss, top1, top5 = 0.0, 0.0, 0.0

    pbar = tqdm(loader, desc=epoch_label, leave=False)
    with torch.set_grad_enabled(training):
        for images, labels in pbar:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item()
            top1 += top_k_accuracy(outputs, labels, k=1)
            top5 += top_k_accuracy(outputs, labels, k=5)
            pbar.set_postfix(loss=f"{loss.item():.4f}")

    n = len(loader)
    return total_loss / n, top1 / n, top5 / n


# === Track running best across trials ===
best_top1, best_top5 = -1.0, -1.0
best_top1_run_id, best_top5_run_id = None, None


# === Optuna objective: one full training run per call ===
def objective(trial):
    global best_top1, best_top5, best_top1_run_id, best_top5_run_id

    # Sample hyperparameters for this trial
    n_warmup    = trial.suggest_int("n_warmup_epochs", 1, 5)
    lr_head     = trial.suggest_float("lr_head", 1e-4, 1e-2, log=True)
    lr_backbone = trial.suggest_float("lr_backbone", 1e-5, 1e-3, log=True)
    n_finetune  = TOTAL_EPOCHS - n_warmup

    with mlflow.start_run(run_name=f"trial_{trial.number}"):
        run_id = mlflow.active_run().info.run_id

        mlflow.log_params({
            "trial_number":      trial.number,
            "n_warmup_epochs":   n_warmup,
            "n_finetune_epochs": n_finetune,
            "lr_head":           lr_head,
            "lr_backbone":       lr_backbone,
            "total_epochs":      TOTAL_EPOCHS,
            "batch_size":        BATCH_SIZE,
            "weight_decay":      WEIGHT_DECAY,
            "optimizer":         "RMSprop",
            "scheduler":         "StepLR",
        })

        model = build_fresh_model()
        train_losses, val_losses = [], []
        final_val_top1 = final_val_top5 = 0.0

        # --- Stage 1: Warmup (backbone frozen, head only trains) ---
        for p in model.features.parameters():
            p.requires_grad = False

        warmup_opt = optim.RMSprop(
            filter(lambda p: p.requires_grad, model.parameters()),
            lr=lr_head, momentum=RMS_MOMENTUM, alpha=RMS_ALPHA,
            eps=RMS_EPS, weight_decay=WEIGHT_DECAY,
        )

        for ep in range(1, n_warmup + 1):
            train_loss, train_top1, train_top5 = run_epoch(
                model, warmup_opt, train_loader, True,
                f"Trial {trial.number} warmup {ep}/{n_warmup}",
            )
            val_loss, val_top1, val_top5 = run_epoch(
                model, warmup_opt, val_loader, False,
                f"Trial {trial.number} val",
            )
            train_losses.append(train_loss)
            val_losses.append(val_loss)
            final_val_top1, final_val_top5 = val_top1, val_top5

            mlflow.log_metrics({
                "train_loss":       train_loss,
                "train_top1":       train_top1,
                "train_top5":       train_top5,
                "val_loss":         val_loss,
                "val_top1":         val_top1,
                "val_top5":         val_top5,
                "val_accuracy_pct": val_top1 * 100,
            }, step=ep)

        # --- Stage 2: Fine-tuning (everything unfrozen, two LRs) ---
        for p in model.features.parameters():
            p.requires_grad = True

        finetune_opt = optim.RMSprop(
            [
                {"params": model.classifier.parameters(), "lr": lr_head},
                {"params": model.features.parameters(),   "lr": lr_backbone},
            ],
            momentum=RMS_MOMENTUM, alpha=RMS_ALPHA,
            eps=RMS_EPS, weight_decay=WEIGHT_DECAY,
        )

        step_size = max(1, n_finetune // 3)
        scheduler = optim.lr_scheduler.StepLR(finetune_opt, step_size=step_size, gamma=0.1)

        for ep in range(1, n_finetune + 1):
            global_ep = n_warmup + ep
            train_loss, train_top1, train_top5 = run_epoch(
                model, finetune_opt, train_loader, True,
                f"Trial {trial.number} finetune {ep}/{n_finetune}",
            )
            val_loss, val_top1, val_top5 = run_epoch(
                model, finetune_opt, val_loader, False,
                f"Trial {trial.number} val",
            )
            scheduler.step()
            train_losses.append(train_loss)
            val_losses.append(val_loss)
            final_val_top1, final_val_top5 = val_top1, val_top5

            mlflow.log_metrics({
                "train_loss":       train_loss,
                "train_top1":       train_top1,
                "train_top5":       train_top5,
                "val_loss":         val_loss,
                "val_top1":         val_top1,
                "val_top5":         val_top5,
                "val_accuracy_pct": val_top1 * 100,
            }, step=global_ep)

        # --- Plot loss curve and log as MLflow artifact ---
        epochs_range = range(1, TOTAL_EPOCHS + 1)
        fig, ax = plt.subplots(figsize=(10, 5))
        ax.plot(epochs_range, train_losses, marker="o", label="Train loss")
        ax.plot(epochs_range, val_losses,   marker="o", label="Validation loss")
        ax.axvline(n_warmup + 0.5, color="gray", linestyle="--", alpha=0.5,
                   label="Warmup → Fine-tune")
        ax.set_xlabel("Epoch")
        ax.set_ylabel("Average loss")
        ax.set_title(f"Trial {trial.number} | val_top1={final_val_top1:.4f}, val_top5={final_val_top5:.4f}")
        ax.legend()
        ax.grid(alpha=0.3)
        fig.tight_layout()
        mlflow.log_figure(fig, "loss_curve.png")
        plt.close(fig)

        # --- Save model if it beat the running best ---
        if final_val_top1 > best_top1:
            best_top1 = final_val_top1
            best_top1_run_id = run_id
            torch.save(model.state_dict(), MODELS_DIR / "EN_best_top1.pth")
            print(f"  → New best top-1: {final_val_top1:.4f} (saved EN_best_top1.pth)")

        if final_val_top5 > best_top5:
            best_top5 = final_val_top5
            best_top5_run_id = run_id
            torch.save(model.state_dict(), MODELS_DIR / "EN_best_top5.pth")
            print(f"  → New best top-5: {final_val_top5:.4f} (saved EN_best_top5.pth)")

        trial.set_user_attr("val_top5", final_val_top5)
        trial.set_user_attr("run_id", run_id)

        return final_val_top1  # Optuna maximizes top-1


# === Run the study ===
mlflow.set_experiment("bird_finetuning")
study = optuna.create_study(direction="maximize", study_name="bird_finetuning")
study.optimize(objective, n_trials=N_TRIALS)

# === Tag the best MLflow runs ===
client = mlflow.MlflowClient()
if best_top1_run_id:
    client.set_tag(best_top1_run_id, "best_top1", "true")
if best_top5_run_id:
    client.set_tag(best_top5_run_id, "best_top5", "true")

# === Summary ===
print("\n" + "=" * 60)
print(f"Best top-1: {best_top1:.4f}  →  EN_best_top1.pth  (run {best_top1_run_id})")
print(f"Best top-5: {best_top5:.4f}  →  EN_best_top5.pth  (run {best_top5_run_id})")
print(f"\nBest hyperparameters (by top-1):")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")

## 3. Evaluate Best Models on Test Set

Loads `EN_best_top1.pth` and `EN_best_top5.pth` and evaluates each on the held-out test set.

In [ ]:
def evaluate_checkpoint(checkpoint_path, loader, label):
    """Load model state from a checkpoint and evaluate on the test set."""
    model = build_fresh_model()
    state_dict = torch.load(checkpoint_path, map_location=device, weights_only=True)
    model.load_state_dict(state_dict)

    test_loss, test_top1, test_top5 = run_epoch(
        model, optimizer=None, loader=loader,
        training=False, epoch_label=f"Test [{label}]",
    )

    print(f"\n=== {label} ({checkpoint_path.name}) ===")
    print(f"Test loss:      {test_loss:.4f}")
    print(f"Test Top-1 acc: {test_top1:.4f}  ({test_top1 * 100:.2f}%)")
    print(f"Test Top-5 acc: {test_top5:.4f}  ({test_top5 * 100:.2f}%)")

    return test_loss, test_top1, test_top5


top1_results = evaluate_checkpoint(MODELS_DIR / "EN_best_top1.pth", "Best Top-1 model")
top5_results = evaluate_checkpoint(MODELS_DIR / "EN_best_top5.pth", "Best Top-5 model")